# Clustering et projections linéaires et non linéaires

## Introduction

Dans ce TP, vous regroupez des images de chiffres manuscrits avec k-means sans regarder leurs étiquettes, puis vous mesurez, grâce aux étiquettes, ce que les clusters ont retrouvé. Vous recommencez ensuite sur des projections des images (ACP, UMAP, t-SNE), et avec deux autres algorithmes : la CAH et DBSCAN.

Les étapes :

1. entraîner k-means et choisir $k$ ;
2. lire les centroïdes ;
3. associer chaque cluster à une classe ;
4. évaluer le clustering ;
5. clusteriser des projections ;
6. CAH sur l'ACP ;
7. DBSCAN et HDBSCAN sur UMAP ;
8. conclure.

Chaque partie s'appuie sur les précédentes : faites-les dans l'ordre. Quand les cellules suivantes appellent une fonction que vous écrivez, sa signature est déjà dans la cellule de l'exercice. Les algorithmes aléatoires prennent tous `random_state=0` : vos chiffres doivent être ceux des conseils, à peu près seulement pour UMAP et t-SNE, dont les résultats varient un peu d'une version à l'autre.

In [ ]:
import matplotlib.pyplot as plt
import numpy
import pandas
import scipy.cluster.hierarchy
import scipy.optimize
import scipy.stats
import seaborn
import sklearn.cluster
import sklearn.datasets
import sklearn.decomposition
import sklearn.manifold
import sklearn.metrics
import sklearn.neighbors
import umap

## Les données

Le jeu *digits* de scikit-learn (UCI, *Optical Recognition of Handwritten Digits*) : 1 797 images de chiffres manuscrits de 8 × 8 pixels. Chaque image est mise à plat en un vecteur de 64 entiers, de 0 (blanc) à 16 (noir), et porte son étiquette, le chiffre de 0 à 9 : 10 classes à peu près équilibrées, de 174 à 183 images chacune. Le clustering n'utilise que les pixels (`chiffres.data`, une ligne par image) ; les étiquettes (`chiffres.target`) servent seulement à l'évaluer.

[Documentation de `load_digits`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)

In [ ]:
chiffres = sklearn.datasets.load_digits()
print(chiffres.data.shape)  # (images, pixels)
print(chiffres.target.shape)
seaborn.countplot(x=chiffres.target)
plt.show()

In [ ]:
# 25 images tirées au hasard, sans remise (pour ne pas afficher deux fois la même)
rng = numpy.random.default_rng(0)
indices = rng.choice(len(chiffres.data), size=25, replace=False)

fig, axes = plt.subplots(5, 5, figsize=(8, 8))
for ax, index in zip(axes.flat, indices):
  # Le vecteur de 64 pixels redevient une image de 8 × 8
  ax.imshow(chiffres.data[index].reshape(8, 8), cmap="gray_r")
  ax.set_title(f"n° {index} : {chiffres.target[index]}")
  ax.axis("off")
plt.show()

## 1. Entraîner k-means et choisir k

Complétez la fonction `entraine` : elle entraîne un k-means à `k` clusters sur `donnees` (une ligne par point), avec plusieurs initialisations et une graine fixée, et rend le modèle entraîné.

Choisissez ensuite $k$ comme dans la frame « Choisir k » du cours : pour $k$ de 2 à 15, entraînez un k-means sur `chiffres.data` et calculez son inertie et sa silhouette moyenne ; tracez les deux courbes en fonction de $k$. Voit-on un coude ? La silhouette la plus haute tombe-t-elle sur 10, le nombre de chiffres ?

Enfin, entraînez le k-means à 10 clusters dans `kmeans`, et rangez dans `clusters` le cluster de chaque image.

**Documentation utile**

- [`sklearn.cluster.KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
- [`sklearn.metrics.silhouette_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html)

In [ ]:
def entraine(donnees: numpy.ndarray, k: int = 10) -> sklearn.cluster.KMeans: ...


# Votre code ici

### Conseils supplémentaires

- `sklearn.cluster.KMeans(n_clusters=k, n_init=10, random_state=0).fit(donnees)` : `n_init=10` lance 10 initialisations (k-means++) et garde celle de plus faible inertie ; `random_state=0` fixe la graine ; `fit` rend le modèle lui-même.
- Après `fit`, `modele.inertia_` est l'inertie, et `modele.labels_` le cluster de chaque point d'entraînement (ce que rendrait `modele.predict(donnees)`).
- `sklearn.metrics.silhouette_score(donnees, etiquettes)` : la silhouette moyenne, de −1 à 1.
- Valeurs attendues : pour $k$ = 10, une inertie d'environ 1 165 000 et une silhouette d'environ 0,18. Les 14 entraînements prennent quelques dizaines de secondes.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def entraine(donnees: numpy.ndarray, k: int = 10) -> sklearn.cluster.KMeans:
  return sklearn.cluster.KMeans(n_clusters=..., n_init=10, random_state=0).fit(...)


ks = range(2, 16)
inerties = []
silhouettes = []
for k in ks:
  modele = entraine(chiffres.data, k)
  inerties.append(...)
  silhouettes.append(sklearn.metrics.silhouette_score(..., ...))

fig, (ax_inertie, ax_silhouette) = plt.subplots(1, 2, figsize=(10, 4))
ax_inertie.plot(ks, inerties, marker="o")
ax_inertie.set(xlabel="k", title="Inertie")
ax_silhouette.plot(ks, silhouettes, marker="o")
ax_silhouette.set(xlabel="k", title="Silhouette moyenne")
plt.show()

kmeans = entraine(...)
clusters = ...
```

### Solution

In [ ]:
def entraine(donnees: numpy.ndarray, k: int = 10) -> sklearn.cluster.KMeans:
  return sklearn.cluster.KMeans(n_clusters=k, n_init=10, random_state=0).fit(donnees)


ks = range(2, 16)
inerties = []
silhouettes = []
for k in ks:
  modele = entraine(chiffres.data, k)
  inerties.append(modele.inertia_)
  silhouettes.append(sklearn.metrics.silhouette_score(chiffres.data, modele.labels_))

fig, (ax_inertie, ax_silhouette) = plt.subplots(1, 2, figsize=(10, 4))
ax_inertie.plot(ks, inerties, marker="o")
ax_inertie.set(xlabel="k", title="Inertie")
ax_silhouette.plot(ks, silhouettes, marker="o")
ax_silhouette.set(xlabel="k", title="Silhouette moyenne")
plt.show()

kmeans = entraine(chiffres.data)
clusters = kmeans.labels_
print(f"Inertie pour k = 10 : {kmeans.inertia_:.0f}")
print(clusters[:10])

L'inertie décroît régulièrement, sans coude net : elle baisse toujours quand $k$ augmente. La silhouette monte jusqu'à $k$ = 9 (0,19) puis reste presque plate jusqu'à 14 : sur les pixels bruts, 10 ne ressort pas. Les deux critères donnent une plage de valeurs ; c'est la connaissance du problème (10 chiffres) qui fixe $k$ = 10.

## 2. Lire les centroïdes

Affichez l'image de chacun des 10 centroïdes, avec son numéro de cluster. Un centroïde est la moyenne des images de son cluster (`kmeans.cluster_centers_`, un vecteur de 64 pixels par cluster) : quel chiffre chaque cluster représente-t-il ? Un chiffre a-t-il deux clusters, un autre aucun ? Un centroïde mélange-t-il deux chiffres ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `kmeans.cluster_centers_.reshape(-1, 8, 8)` : les 10 centroïdes en images de 8 × 8 (avec `-1`, numpy calcule cette dimension).
- `plt.subplots(1, 10, figsize=(10, 2))` donne une rangée de 10 axes ; sur chacun, `ax.imshow(image, cmap="gray_r")`, `ax.set_title(numéro)` et `ax.axis("off")`, comme dans la grille d'exemples plus haut.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
centres = kmeans.cluster_centers_.reshape(...)
fig, axes = plt.subplots(1, 10, figsize=(10, 2))
for cluster, (ax, centre) in enumerate(zip(axes, centres)):
  ax.imshow(..., cmap="gray_r")
  ax.set_title(cluster)
  ax.axis("off")
plt.show()
```

### Solution

In [ ]:
centres = kmeans.cluster_centers_.reshape(-1, 8, 8)
fig, axes = plt.subplots(1, 10, figsize=(10, 2))
for cluster, (ax, centre) in enumerate(zip(axes, centres)):
  ax.imshow(centre, cmap="gray_r")
  ax.set_title(cluster)
  ax.axis("off")
plt.show()

La plupart des centroïdes se lisent comme un chiffre net : 0, 2, 3, 4, 5, 6, 7, 9. Deux centroïdes ressemblent à un 1, et aucun à un 8 : l'un des deux « 1 » est flou, son cluster contient autant de 8 que de 1. Les numéros des clusters sont arbitraires : le cluster 0 n'est pas forcément le chiffre 0.

## 3. Associer chaque cluster à une classe

Les numéros de clusters sont arbitraires : pour comparer les clusters aux étiquettes avec une mesure de classification comme l'accuracy, il faut d'abord associer une classe à chaque cluster, le *mapping* de la frame « Avec des étiquettes » du cours. Deux façons de le faire, une par exercice.

### 3.1 Classe majoritaire

Complétez `predit_mode` : pour chaque point, elle rend la classe la plus représentée (le mode) parmi les étiquettes `cibles` des points de son cluster. Elle prend les clusters et non le modèle, pour servir à n'importe quel algorithme de clustering. Rangez son résultat sur les clusters du k-means dans `predictions_mode`.

**Documentation utile**

- [`scipy.stats.mode`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mode.html)

In [ ]:
def predit_mode(clusters: numpy.ndarray, cibles: numpy.ndarray) -> numpy.ndarray: ...


# Votre code ici

#### Conseils supplémentaires

- `numpy.unique(clusters)` : les numéros de clusters ; `clusters == cluster` : le masque des points d'un cluster ; `cibles[masque]` : leurs étiquettes.
- `scipy.stats.mode(values).mode` : la valeur la plus fréquente (la plus petite en cas d'égalité).
- `numpy.empty_like(clusters)` : un tableau de la forme de `clusters`, à remplir.
- Exemple à vérifier à la main : `predit_mode(numpy.array([0, 0, 0, 1, 1, 1]), numpy.array([0, 0, 0, 0, 0, 1]))` rend `[0 0 0 0 0 0]` : la classe 0 est majoritaire dans les deux clusters.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def predit_mode(clusters: numpy.ndarray, cibles: numpy.ndarray) -> numpy.ndarray:
  predictions = numpy.empty_like(clusters)
  for cluster in numpy.unique(clusters):
    masque = ...
    predictions[masque] = scipy.stats.mode(...).mode
  return predictions


print(predit_mode(numpy.array([0, 0, 0, 1, 1, 1]), numpy.array([0, 0, 0, 0, 0, 1])))
predictions_mode = predit_mode(..., ...)
```

#### Solution

In [ ]:
def predit_mode(clusters: numpy.ndarray, cibles: numpy.ndarray) -> numpy.ndarray:
  predictions = numpy.empty_like(clusters)
  for cluster in numpy.unique(clusters):
    masque = clusters == cluster
    predictions[masque] = scipy.stats.mode(cibles[masque]).mode
  return predictions


# La même chose avec pandas, selon le schéma « split, apply, combine »
def predit_mode_avec_pandas(
  clusters: numpy.ndarray, cibles: numpy.ndarray
) -> numpy.ndarray:
  return (
    pandas.Series(cibles).groupby(clusters).transform(lambda x: x.mode()[0]).to_numpy()
  )


print(predit_mode(numpy.array([0, 0, 0, 1, 1, 1]), numpy.array([0, 0, 0, 0, 0, 1])))
predictions_mode = predit_mode(clusters, chiffres.target)

### 3.2 Algorithme hongrois

Le mode peut donner la même classe à deux clusters, et n'en donner aucun à une autre classe. L'algorithme hongrois cherche au contraire une association un pour un, une classe par cluster, qui maximise le nombre de points bien classés. Complétez `predit_hongrois`, de même signature, avec la matrice de contingence (le nombre de points de chaque classe dans chaque cluster) et `linear_sum_assignment`. Elle suppose autant de clusters que de classes, numérotés à partir de 0. Rangez son résultat sur les clusters du k-means dans `predictions`.

**Documentation utile**

- [`sklearn.metrics.cluster.contingency_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.cluster.contingency_matrix.html)
- [`scipy.optimize.linear_sum_assignment`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linear_sum_assignment.html)

In [ ]:
def predit_hongrois(
  clusters: numpy.ndarray, cibles: numpy.ndarray
) -> numpy.ndarray: ...


# Votre code ici

#### Conseils supplémentaires

- `contingency_matrix(cibles, clusters)` : la matrice `contingence` où `contingence[c, k]` est le nombre de points de la classe `c` dans le cluster `k`.
- `linear_sum_assignment(contingence, maximize=True)` rend deux tableaux, `classes` et `ids_clusters` : la classe `classes[i]` va au cluster `ids_clusters[i]`, et la somme des `contingence[classes[i], ids_clusters[i]]` est la plus grande possible (sans `maximize=True`, il cherche la plus petite, comme pour un coût).
- Il reste à inverser cette association : `correspondance[ids_clusters] = classes` remplit un tableau `correspondance` où `correspondance[k]` est la classe du cluster `k` ; `correspondance[clusters]` donne alors la classe de chaque point.
- Exemple : avec les mêmes tableaux qu'en 3.1, `predit_hongrois` rend `[0 0 0 1 1 1]` : le cluster 1 reçoit la classe 1, que le mode ne prédisait jamais.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def predit_hongrois(clusters: numpy.ndarray, cibles: numpy.ndarray) -> numpy.ndarray:
  contingence = sklearn.metrics.cluster.contingency_matrix(..., ...)
  classes, ids_clusters = scipy.optimize.linear_sum_assignment(
    contingence, maximize=True
  )
  correspondance = numpy.empty_like(ids_clusters)
  correspondance[...] = ...
  return correspondance[...]


print(predit_hongrois(numpy.array([0, 0, 0, 1, 1, 1]), numpy.array([0, 0, 0, 0, 0, 1])))
predictions = predit_hongrois(..., ...)
```

#### Solution

In [ ]:
def predit_hongrois(clusters: numpy.ndarray, cibles: numpy.ndarray) -> numpy.ndarray:
  contingence = sklearn.metrics.cluster.contingency_matrix(cibles, clusters)
  classes, ids_clusters = scipy.optimize.linear_sum_assignment(
    contingence, maximize=True
  )
  correspondance = numpy.empty_like(ids_clusters)
  correspondance[ids_clusters] = classes
  return correspondance[clusters]


print(predit_hongrois(numpy.array([0, 0, 0, 1, 1, 1]), numpy.array([0, 0, 0, 0, 0, 1])))
predictions = predit_hongrois(clusters, chiffres.target)

# La classe donnée à chaque cluster par chacune des deux méthodes
for nom, preds in [("mode", predictions_mode), ("hongrois", predictions)]:
  classes = [int(preds[clusters == k][0]) for k in range(10)]
  print(f"Classes des clusters 0 à 9 ({nom}) : {classes}")

## 4. Évaluer le clustering

Deux familles de mesures (frames « Avec des étiquettes » et « Silhouette » du cours) : celles qui comparent les classes associées aux étiquettes, puis celles qui se passent du mapping.

### 4.1 Avec les étiquettes : accuracy et matrice de confusion

Calculez l'accuracy des deux associations (`predictions_mode` et `predictions`), puis affichez la matrice de confusion entre les étiquettes et `predictions`. L'une des deux associations a-t-elle une meilleure accuracy ? Laquelle donne l'image la plus juste du clustering ? Quels chiffres k-means confond-il ?

**Documentation utile**

- [`sklearn.metrics.accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
- [`sklearn.metrics.ConfusionMatrixDisplay.from_predictions`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_predictions)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `sklearn.metrics.accuracy_score(chiffres.target, predictions)` : la part des points dont la classe associée est la bonne.
- `sklearn.metrics.ConfusionMatrixDisplay.from_predictions(chiffres.target, predictions)` trace la matrice (lignes : étiquettes ; colonnes : classes prédites), puis `plt.show()` l'affiche.
- Valeurs attendues : une accuracy d'environ 0,79 pour les deux associations.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
justesse_mode = sklearn.metrics.accuracy_score(chiffres.target, ...)
justesse_hongrois = sklearn.metrics.accuracy_score(..., ...)
print(f"Accuracy (mode) : {justesse_mode:.3f}")
print(f"Accuracy (hongrois) : {justesse_hongrois:.3f}")

sklearn.metrics.ConfusionMatrixDisplay.from_predictions(..., ...)
plt.show()
```

#### Solution

In [ ]:
justesse_mode = sklearn.metrics.accuracy_score(chiffres.target, predictions_mode)
justesse_hongrois = sklearn.metrics.accuracy_score(chiffres.target, predictions)
print(f"Accuracy (mode) : {justesse_mode:.3f}")
print(f"Accuracy (hongrois) : {justesse_hongrois:.3f}")

sklearn.metrics.ConfusionMatrixDisplay.from_predictions(chiffres.target, predictions)
plt.show()

Les deux associations ont ici la même accuracy (0,79) : le cluster qui mélange les 1 et les 8 en contient autant de chaque, et le mode, à égalité, prend la plus petite classe. Mais le mode donne deux clusters au 1 et aucun au 8 : il ne prédit jamais 8, ce que l'accuracy ne montre pas. En général, l'accuracy du mode est au moins celle du hongrois, puisque chaque cluster y prend sa meilleure classe sans contrainte, et elle gonfle avec le nombre de clusters (un cluster par point : 1). Le hongrois, un pour un, donne l'image la plus juste.

La matrice de confusion montre les erreurs de k-means : plus de la moitié des 1 tombent dans le cluster associé au 8, et une partie des 5, des 8 et des 1 est prise pour des 9 ou des 1.

### 4.2 Sans mapping : ARI, AMI et silhouette

Complétez `evalue` : elle rend trois scores du clustering `clusters`, qui se passent de mapping :

- l'ARI et l'AMI (frame « Avec des étiquettes ») comparent les clusters aux étiquettes `cibles` : 1 pour une partition identique, environ 0 pour un clustering au hasard ;
- la silhouette moyenne n'utilise pas les étiquettes, mais les distances entre les points de `donnees`.

Passez-lui toujours les données d'origine (`chiffres.data`), même quand le clustering est fait sur une projection (parties 5 à 7) : la silhouette dépend des distances de l'espace où on la calcule, et deux silhouettes calculées dans deux espaces différents ne se comparent pas. Calculées toutes sur les 64 pixels, elles disent quel clustering regroupe le mieux les images proches pixel à pixel.

Rangez le résultat du k-means dans le dictionnaire `resultats`, sous la clé `"k-means"`, et affichez-le en tableau : les parties suivantes y ajoutent les leurs.

**Documentation utile**

- [`sklearn.metrics.adjusted_rand_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html)
- [`sklearn.metrics.adjusted_mutual_info_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_mutual_info_score.html)

In [ ]:
def evalue(
  clusters: numpy.ndarray, cibles: numpy.ndarray, donnees: numpy.ndarray
) -> dict[str, float]: ...


# Votre code ici

#### Conseils supplémentaires

- `sklearn.metrics.adjusted_rand_score(cibles, clusters)`, `sklearn.metrics.adjusted_mutual_info_score(cibles, clusters)`, `sklearn.metrics.silhouette_score(donnees, clusters)`.
- Les clés du dictionnaire rendu deviennent les colonnes du tableau : `{"ARI": ..., "AMI": ..., "silhouette": ...}`.
- `pandas.DataFrame(resultats).T` présente `resultats`, un dictionnaire de dictionnaires, avec une ligne par méthode.
- Valeurs attendues pour le k-means : ARI ≈ 0,67, AMI ≈ 0,74, silhouette ≈ 0,18.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def evalue(
  clusters: numpy.ndarray, cibles: numpy.ndarray, donnees: numpy.ndarray
) -> dict[str, float]:
  return {
    "ARI": sklearn.metrics.adjusted_rand_score(..., ...),
    "AMI": ...,
    "silhouette": sklearn.metrics.silhouette_score(..., ...),
  }


resultats = {"k-means": evalue(clusters, chiffres.target, ...)}
pandas.DataFrame(resultats).T.round(2)
```

#### Solution

In [ ]:
def evalue(
  clusters: numpy.ndarray, cibles: numpy.ndarray, donnees: numpy.ndarray
) -> dict[str, float]:
  return {
    "ARI": sklearn.metrics.adjusted_rand_score(cibles, clusters),
    "AMI": sklearn.metrics.adjusted_mutual_info_score(cibles, clusters),
    "silhouette": sklearn.metrics.silhouette_score(donnees, clusters),
  }


resultats = {"k-means": evalue(clusters, chiffres.target, chiffres.data)}
pandas.DataFrame(resultats).T.round(2)

## 5. Clusteriser des projections

Sur les 64 pixels, la distance euclidienne entre deux images mesure mal leur ressemblance. Projetez les images dans un espace plus petit, puis entraînez sur chaque projection un k-means à 10 clusters (`entraine`) et évaluez-le (`evalue`, toujours avec `chiffres.data` en troisième argument) :

- une ACP qui garde 80 % de la variance, dans `proj_acp` ;
- UMAP en 2 dimensions, dans `proj_umap` ;
- t-SNE en 2 dimensions, dans `proj_tsne`.

Ajoutez chaque évaluation à `resultats` et affichez le tableau. Tracez aussi les projections UMAP et t-SNE, chaque point coloré par sa classe : les chiffres y sont-ils séparés ? Comment l'ARI évolue-t-il, et la silhouette ?

**Documentation utile**

- [`sklearn.decomposition.PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [`umap.UMAP`](https://umap-learn.readthedocs.io/en/latest/api.html)
- [`sklearn.manifold.TSNE`](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `sklearn.decomposition.PCA(n_components=0.8)` : un nombre entre 0 et 1 garde autant de composantes qu'il en faut pour expliquer cette part de la variance (13 ici).
- `umap.UMAP(random_state=0, n_jobs=1)` et `sklearn.manifold.TSNE(random_state=0)` projettent en 2 dimensions par défaut. Avec une graine, UMAP ne calcule pas en parallèle : `n_jobs=1` le dit, sans quoi il prévient. Comptez de quelques dizaines de secondes à une minute pour chacun.
- `.fit_transform(chiffres.data)` rend la projection, une ligne par image.
- `ax.scatter(x, y, c=chiffres.target, cmap="tab10", s=5)` colore chaque point par sa classe ; `fig.colorbar(points, ax=axes)` ajoute la légende des couleurs.
- Valeurs attendues (ARI) : environ 0,66 sur l'ACP, 0,82 sur UMAP et 0,89 sur t-SNE. La silhouette, elle, reste entre 0,16 et 0,18.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
proj_acp = sklearn.decomposition.PCA(n_components=...).fit_transform(chiffres.data)
proj_umap = umap.UMAP(random_state=0, n_jobs=1).fit_transform(...)
proj_tsne = sklearn.manifold.TSNE(random_state=0).fit_transform(...)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (nom, proj) in zip(axes, [("UMAP", proj_umap), ("t-SNE", proj_tsne)]):
  points = ax.scatter(..., ..., c=chiffres.target, cmap="tab10", s=5)
  ax.set_title(nom)
fig.colorbar(points, ax=axes)
plt.show()

for nom, proj in [
  ("ACP + k-means", proj_acp),
  ("UMAP + k-means", proj_umap),
  ("t-SNE + k-means", proj_tsne),
]:
  modele = entraine(...)
  resultats[nom] = evalue(modele.labels_, chiffres.target, ...)
pandas.DataFrame(resultats).T.round(2)
```

### Solution

In [ ]:
proj_acp = sklearn.decomposition.PCA(n_components=0.8).fit_transform(chiffres.data)
proj_umap = umap.UMAP(random_state=0, n_jobs=1).fit_transform(chiffres.data)
proj_tsne = sklearn.manifold.TSNE(random_state=0).fit_transform(chiffres.data)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (nom, proj) in zip(axes, [("UMAP", proj_umap), ("t-SNE", proj_tsne)]):
  points = ax.scatter(proj[:, 0], proj[:, 1], c=chiffres.target, cmap="tab10", s=5)
  ax.set_title(nom)
fig.colorbar(points, ax=axes)
plt.show()

for nom, proj in [
  ("ACP + k-means", proj_acp),
  ("UMAP + k-means", proj_umap),
  ("t-SNE + k-means", proj_tsne),
]:
  modele = entraine(proj)
  resultats[nom] = evalue(modele.labels_, chiffres.target, chiffres.data)
pandas.DataFrame(resultats).T.round(2)

Sur UMAP et t-SNE, les chiffres forment des groupes compacts et séparés : k-means les retrouve bien mieux (ARI 0,82 et 0,89, contre 0,67 sur les pixels). L'ACP, linéaire, ne change presque rien (0,66). La silhouette sur les pixels, elle, baisse un peu : les clusters suivent mieux les chiffres, pas les distances entre pixels (la conclusion y revient).

## 6. CAH sur l'ACP

La classification hiérarchique sur composantes principales du cours (« Méthodes hybrides ») : une CAH, critère de Ward, sur `proj_acp`, où les distances sont moins bruitées que sur les pixels. Tracez d'abord le dendrogramme, et lisez à quelle hauteur le couper pour obtenir 10 clusters : suggère-t-il un autre nombre de clusters ? Entraînez ensuite une CAH à 10 clusters, évaluez-la et ajoutez-la à `resultats` : fait-elle mieux que k-means sur la même ACP ?

**Documentation utile**

- [`scipy.cluster.hierarchy.linkage`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html)
- [`scipy.cluster.hierarchy.dendrogram`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.dendrogram.html)
- [`sklearn.cluster.AgglomerativeClustering`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `scipy.cluster.hierarchy.linkage(proj_acp, method="ward")` rend la matrice des fusions, une ligne par fusion dans l'ordre, la hauteur de la fusion dans la colonne 2 : `fusions[-1, 2]` est la hauteur de la dernière (de 2 clusters à 1).
- Pour 10 clusters, on défait les 9 dernières fusions : la coupe passe entre `fusions[-10, 2]` et `fusions[-9, 2]`.
- `scipy.cluster.hierarchy.dendrogram(fusions, truncate_mode="lastp", p=30, ax=ax)` ne trace que les 30 derniers clusters (sinon, 1 797 feuilles) ; `ax.axhline(hauteur, color="red")` trace la coupe.
- `sklearn.cluster.AgglomerativeClustering(n_clusters=10, linkage="ward").fit_predict(proj_acp)` : la même CAH que scipy, coupée à 10 clusters. Elle n'est pas aléatoire : pas de graine.
- Valeurs attendues : une coupe entre 260 et 277 environ ; un ARI d'environ 0,74.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
fusions = scipy.cluster.hierarchy.linkage(..., method="ward")
coupe = (fusions[-10, 2] + fusions[-9, 2]) / 2
print(f"Coupe pour 10 clusters : entre {fusions[-10, 2]:.0f} et {fusions[-9, 2]:.0f}")

fig, ax = plt.subplots(figsize=(12, 4))
scipy.cluster.hierarchy.dendrogram(fusions, truncate_mode="lastp", p=30, ax=ax)
ax.axhline(coupe, color="red")
plt.show()

cah = sklearn.cluster.AgglomerativeClustering(n_clusters=..., linkage=...)
resultats["ACP + CAH"] = evalue(cah.fit_predict(...), chiffres.target, chiffres.data)
pandas.DataFrame(resultats).T.round(2)
```

### Solution

In [ ]:
fusions = scipy.cluster.hierarchy.linkage(proj_acp, method="ward")
coupe = (fusions[-10, 2] + fusions[-9, 2]) / 2
print(f"Coupe pour 10 clusters : entre {fusions[-10, 2]:.0f} et {fusions[-9, 2]:.0f}")

fig, ax = plt.subplots(figsize=(12, 4))
scipy.cluster.hierarchy.dendrogram(fusions, truncate_mode="lastp", p=30, ax=ax)
ax.axhline(coupe, color="red")
plt.show()

cah = sklearn.cluster.AgglomerativeClustering(n_clusters=10, linkage="ward")
resultats["ACP + CAH"] = evalue(
  cah.fit_predict(proj_acp), chiffres.target, chiffres.data
)
pandas.DataFrame(resultats).T.round(2)

Pour 10 clusters, la coupe passe entre 260 et 277 environ (ligne rouge), un intervalle étroit : les fusions de cette zone se suivent de près. Hors de la dernière fusion, le plus grand saut, de 310 à 383 environ, suggérerait plutôt 7 clusters. La CAH fait mieux que k-means sur la même ACP (ARI 0,74 contre 0,66), sans aléa ; mais elle coûte $O(n^2)$ en mémoire (frame « Coût ») : instantanée sur 1 797 images, elle demanderait, sur des millions, la classification mixte du cours.

## 7. DBSCAN et HDBSCAN sur UMAP

Sur la projection UMAP, les chiffres forment des groupes denses séparés par du vide : le terrain de DBSCAN, qui n'a pas besoin du nombre de clusters. Choisissez $\epsilon$ par la courbe des k-distances du cours (frame « DBSCAN — Limites ») : la distance de chaque point à son `min_samples`-ième voisin, triée, avec `min_samples` = 10 ; $\epsilon$ se lit au coude. Entraînez DBSCAN sur `proj_umap`, comptez ses clusters et ses points de bruit (étiquette −1), évaluez-le et ajoutez-le à `resultats`.

Essayez ensuite HDBSCAN, qui n'a pas d'$\epsilon$ à choisir, seulement la taille minimale d'un cluster (`min_cluster_size`), par exemple 50. Les deux font-ils mieux que k-means sur la même projection ? Combien de clusters trouvent-ils, et pourquoi pas 10 ?

**Documentation utile**

- [`sklearn.neighbors.NearestNeighbors`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html)
- [`sklearn.cluster.DBSCAN`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html)
- [`sklearn.cluster.HDBSCAN`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `sklearn.neighbors.NearestNeighbors(n_neighbors=10).fit(proj_umap).kneighbors(proj_umap)` rend `distances, indices`, une ligne par point et une colonne par voisin, du plus proche au plus loin. Le premier voisin est le point lui-même, à distance 0, comme dans le compte de `min_samples` : `numpy.sort(distances[:, -1])` est la courbe des k-distances.
- `sklearn.cluster.DBSCAN(eps=..., min_samples=10).fit_predict(proj_umap)` et `sklearn.cluster.HDBSCAN(min_cluster_size=50, copy=True).fit_predict(proj_umap)` rendent les étiquettes ; aucun des deux n'est aléatoire. `copy=True` (travailler sur une copie des données) fait taire un avertissement des versions récentes de scikit-learn.
- `etiquettes.max() + 1` est le nombre de clusters, `(etiquettes == -1).sum()` le nombre de points de bruit ; `evalue` compte le bruit comme un cluster de plus.
- Valeurs attendues, à peu près seulement (les coordonnées de UMAP varient d'une version à l'autre) : un coude vers 0,35 à 0,4 ; avec $\epsilon$ = 0,4, DBSCAN trouve 17 clusters, 3 points de bruit, et un ARI d'environ 0,89 ; HDBSCAN trouve 9 clusters, 9 points de bruit, et un ARI d'environ 0,82.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
voisins = sklearn.neighbors.NearestNeighbors(n_neighbors=10).fit(proj_umap)
distances, _ = voisins.kneighbors(proj_umap)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(numpy.sort(distances[:, ...]))
ax.set(xlabel="points, triés", ylabel="distance au 10e voisin")
ax.grid()
plt.show()

for nom, modele in [
  ("UMAP + DBSCAN", sklearn.cluster.DBSCAN(eps=..., min_samples=10)),
  ("UMAP + HDBSCAN", sklearn.cluster.HDBSCAN(min_cluster_size=..., copy=True)),
]:
  etiquettes = modele.fit_predict(...)
  print(
    f"{nom} : {etiquettes.max() + 1} clusters, {(etiquettes == -1).sum()} points de bruit"
  )
  resultats[nom] = evalue(etiquettes, chiffres.target, chiffres.data)
pandas.DataFrame(resultats).T.round(2)
```

### Solution

In [ ]:
voisins = sklearn.neighbors.NearestNeighbors(n_neighbors=10).fit(proj_umap)
distances, _ = voisins.kneighbors(proj_umap)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(numpy.sort(distances[:, -1]))
ax.set(xlabel="points, triés", ylabel="distance au 10e voisin")
ax.grid()
plt.show()

# eps : lu au coude de la courbe
for nom, modele in [
  ("UMAP + DBSCAN", sklearn.cluster.DBSCAN(eps=0.4, min_samples=10)),
  ("UMAP + HDBSCAN", sklearn.cluster.HDBSCAN(min_cluster_size=50, copy=True)),
]:
  etiquettes = modele.fit_predict(proj_umap)
  print(
    f"{nom} : {etiquettes.max() + 1} clusters, {(etiquettes == -1).sum()} points de bruit"
  )
  resultats[nom] = evalue(etiquettes, chiffres.target, chiffres.data)
pandas.DataFrame(resultats).T.round(2)

La courbe des k-distances monte lentement, puis se redresse vers 0,35 à 0,4 : au-delà, les points sont isolés. Avec $\epsilon$ = 0,4, DBSCAN fait mieux que k-means sur la même projection (ARI 0,89 contre 0,82) ; il trouve 17 clusters, dont plusieurs petits, de 10 à 30 images : des sous-groupes d'un même chiffre que UMAP a placés à l'écart. Ils coûtent peu à l'ARI, car chacun est pur. HDBSCAN, qui ignore les groupes de moins de 50 images, trouve 9 clusters : il réunit les 1 et les 8, deux groupes qui se touchent dans la projection (ARI 0,82). Ni l'un ni l'autre ne vise 10 clusters : ils suivent la densité, et le nombre de chiffres n'est pas une propriété de la densité.

## 8. Conclusion

- D'après l'ARI, quelle représentation et quel algorithme regroupent le mieux les chiffres ?
- Pourquoi UMAP et t-SNE aident-ils k-means ? Pourquoi la silhouette, calculée sur les pixels, ne suit-elle pas l'ARI ?
- Pour regrouper chaque jour de nouvelles images, utiliseriez-vous t-SNE ? UMAP ?

Répondez dans la cellule suivante, en commentaire.

In [ ]:
# Vos réponses ici, en commentaire

### Conseils supplémentaires

- Relisez la colonne ARI du tableau `resultats`.
- Sur quelles distances k-means travaille-t-il dans chaque cas, et sur lesquelles la silhouette est-elle calculée ? Que vaudrait la silhouette calculée dans l'espace de UMAP ?
- Que peut-on faire d'une nouvelle image avec un t-SNE déjà entraîné ? Cherchez une méthode `transform` dans la documentation de `TSNE`, puis dans celle de `UMAP`.

### Solution

- D'après l'ARI, k-means sur t-SNE et DBSCAN sur UMAP (environ 0,89), puis k-means et HDBSCAN sur UMAP (environ 0,82). Sur les pixels ou l'ACP, k-means plafonne vers 0,67 ; la CAH fait mieux sur la même ACP (0,74).
- UMAP et t-SNE rapprochent les images voisines et écartent les groupes : chaque chiffre devient un nuage compact, la forme que k-means sait trouver. L'ACP, linéaire, garde à peu près les distances entre pixels, et donc les mêmes confusions. La silhouette sur les pixels mesure si les clusters sont compacts pour la distance entre pixels : un clustering qui suit mieux les chiffres ne l'est pas forcément, puisque deux façons d'écrire un même chiffre peuvent être loin pixel à pixel. Calculée dans l'espace de UMAP, elle monterait vers 0,8, mais elle mesurerait alors la projection, faite pour séparer les groupes, et non le clustering : c'est pourquoi on la calcule dans un seul espace pour comparer.
- t-SNE est un outil de visualisation : les distances entre les groupes de sa carte ne veulent rien dire, et il n'a pas de `transform` : une nouvelle image oblige à tout recalculer, et la carte change. Clusteriser sa sortie n'est pas une méthode de production. UMAP a un `transform` pour de nouveaux points ; pour clusteriser, sa documentation conseille `min_dist=0` (des groupes plus denses), et rien n'oblige à rester en 2 dimensions.